[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KUMARSOK/BITS_PAML_AGAI_G10_A1/blob/member2-experiments/notebooks/Member2_Structured_Outputs_Pydantic.ipynb)

# Member 2: Structured Outputs & Flow Engineering (Experiment E4)

This notebook contains the complete technical deliverables for **Member 2** on the BITS Pilani PCAM ZG516 Assignment 1:

- **Programmatic Prompt Templates:** Schema-guided prompts enforcing structured formatting
- **Pydantic Validation Layer (Experiment E4):** Enforcing strict JSON schema adherence and typing
- **Tool / Function Calling:** Exporting validated action items to `src.tools.jira_exporter`
- **Adherence Metrics:** JSON syntax validity, schema error diagnostics, and field completeness

It directly connects with Member 1's dataset subset (`data/experimental_subset.csv`) and quantized model pipeline (`src/model_pipeline.py`).

## 1. Colab Runtime & Workspace Setup

Select **Runtime > Change runtime type > T4 GPU** before running.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

# Clone repo or pull latest updates on Member 2 branch
REPO_URL = "https://github.com/KUMARSOK/BITS_PAML_AGAI_G10_A1.git"
BRANCH = "member2-experiments"
REPO_DIR = Path("/content/BITS_PAML_AGAI_G10_A1")

if Path("/content").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
    os.chdir(REPO_DIR)

print(f"Current working directory: {Path.cwd()}")

In [ ]:
# Install runtime packages needed by model loader and Member 2 module
%pip install -q pydantic jinja2 pandas numpy datasets transformers accelerate bitsandbytes sentencepiece python-dotenv

## 2. Inspecting the Pydantic Schemas & Tool Integration

In [ ]:
import sys
import importlib

# Ensure modules reload if pulled
for mod in ["src.model_pipeline", "src.member2_experiments"]:
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from src.member2_experiments import (
    StructuredMeetingOutput,
    ActionItem,
    TaskPriority,
    STRUCTURED_PROMPT_TEMPLATE,
    FEW_SHOT_STRUCTURED_TEMPLATE,
    render_prompt,
    parse_and_validate,
    run_experiment_e4
)
from src.tools.jira_exporter import export_action_items_to_jira

# Display the generated JSON schema contract
import json
print("Pydantic JSON Schema Contract for StructuredMeetingOutput:")
print(json.dumps(StructuredMeetingOutput.model_json_schema(), indent=2))

## 3. Execute Experiment E4

Runs the structured output evaluation across sample dialogues, verifying JSON syntax and Pydantic validation.

In [ ]:
metrics = run_experiment_e4(
    data_path="data/experimental_subset.csv",
    output_dir="data/member2_results",
    sample_limit=3,
    use_few_shot=False,
    project_key="SAM"
)

print("\n--- Experiment E4 Summary Metrics ---")
print(json.dumps(metrics, indent=2))

## 4. Review Exported Jira Action Items

Inspect the artifacts produced and dispatch to the project tracking tool.

In [ ]:
with open("data/member2_results/e4_structured_outputs.json", "r", encoding="utf-8") as f:
    records = json.load(f)

for r in records:
    print(f"\nSample #{r['sample_id']} - Valid Schema: {r['is_schema_valid']}")
    if r['jira_export_response']:
        print("Jira Dispatch Response:", json.dumps(r['jira_export_response'], indent=2))